Gathering Data

In [18]:
import pandas as pd
import numpy as np

# Load datase
filename = 'synthetic_financial_transactions_2025.csv'
df = pd.read_csv(filename)

print("Dataset berhasil dimuat!")

Dataset berhasil dimuat!


In [19]:
# 1. DATA INSPECTION
print("--- INFO DATASET ---")
df.info()

print("\n--- CEK MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- CEK DUPLICATES ---")
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

# Tampilkan 5 baris pertama
df.head()

--- INFO DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41527 entries, 0 to 41526
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         41527 non-null  object 
 1   date            41527 non-null  object 
 2   amount          40138 non-null  float64
 3   type            41527 non-null  object 
 4   category        41527 non-null  object 
 5   subcategory     40128 non-null  object 
 6   payment_method  40106 non-null  object 
 7   description     40136 non-null  object 
dtypes: float64(1), object(7)
memory usage: 2.5+ MB

--- CEK MISSING VALUES ---
user_id              0
date                 0
amount            1389
type                 0
category             0
subcategory       1399
payment_method    1421
description       1391
dtype: int64

--- CEK DUPLICATES ---
Jumlah baris duplikat: 3894


,user_id,date,amount,type,category,subcategory,payment_method,description
0,USER_005,2025-03-26,NaN,expense,Food,NaN,cash,Pembayaran warung
1,USER_006,2025-11-13,26000.0,expense,Food,warung,debit,Pembayaran warung
2,USER_013,2025-05-14,18500.0,expense,Others,lainnya,e-wallet,Pembayaran lainnya
3,USER_031,2025-01-16,37000.0,expense,Others,lainnya,e-wallet,Pembayaran lainnya
4,USER_046,2025-11-02,25500.0,expense,Others,lainnya,cash,Pembayaran lainnya


### 2. Handling Duplicates
Data transaksi tidak boleh memiliki duplikat yang identik di semua kolom, karena itu bisa berarti transaksi tercatat dua kali (*double charging*). Kita akan menghapus baris duplikat tersebut.

In [20]:
# Menghapus duplikat
df_clean = df.drop_duplicates()

print(f"Total baris sebelum hapus duplikat: {len(df):,}")
print(f"Total baris setelah hapus duplikat: {len(df_clean):,}")
print(f"Sisa duplikat: {df_clean.duplicated().sum()}")

Total baris sebelum hapus duplikat: 41,527
Total baris setelah hapus duplikat: 37,633
Sisa duplikat: 0


### 3. Handling Missing Values (NaN)
Untuk menangani data yang kosong (*missing values*).

In [21]:
print("--- Sebelum Drop Missing Values ---")
print(df_clean.isnull().sum())

# Langsung hapus semua baris yang mengandung NaN
df_clean = df_clean.dropna()

print("\n--- Setelah Drop Missing Values ---")
print(df_clean.isnull().sum())
print(f"\nTotal baris yang tersisa: {len(df_clean):,}")

--- Sebelum Drop Missing Values ---
user_id              0
date                 0
amount            1389
type                 0
category             0
subcategory       1399
payment_method    1421
description       1391
dtype: int64

--- Setelah Drop Missing Values ---
user_id           0
date              0
amount            0
type              0
category          0
subcategory       0
payment_method    0
description       0
dtype: int64

Total baris yang tersisa: 34,607


### 4. Data Type Formatting & 5. Text Standardization
- Memastikan kolom `date` berformat `datetime` bukan `object` (string).
- Memastikan `amount` berformat `integer` agar tidak ada angka desimal yang tidak perlu di transaksi rupiah.
- Menyeragamkan huruf kecil/besar (*case folding*) dan menghapus spasi tersembunyi di akhir/awal kata.

In [22]:
# 4. Format Tipe Data
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['amount'] = df_clean['amount'].astype(int)

# 5. Standardisasi Teks (Case folding & strip)
# str.lower() untuk huruf kecil, str.title() untuk Huruf Kapital Di Awal Kata, str.strip() untuk hapus spasi berlebih
df_clean['payment_method'] = df_clean['payment_method'].astype(str).str.lower().str.strip()
df_clean['category'] = df_clean['category'].astype(str).str.title().str.strip()
df_clean['subcategory'] = df_clean['subcategory'].astype(str).str.lower().str.strip()
df_clean['type'] = df_clean['type'].astype(str).str.lower().str.strip()

print("Tipe data setelah diperbaiki:")
print(df_clean.dtypes)

Tipe data setelah diperbaiki:
user_id                   object
date              datetime64[ns]
amount                     int64
type                      object
category                  object
subcategory               object
payment_method            object
description               object
dtype: object


### 6. Sanity Check & Export Data
Langkah terakhir adalah memastikan tidak ada anomali logika (misal: jumlah uang negatif) dan mengurutkan kembali data berdasarkan User ID dan Tanggal agar rapi saat dianalisis.

In [23]:
# Sanity Check: Mengecek apakah ada amount yang bernilai 0 atau negatif
invalid_amounts = df_clean[df_clean['amount'] <= 0]
if not invalid_amounts.empty:
    print(f"Ditemukan {len(invalid_amounts)} baris dengan amount tidak valid (<= 0). Menghapus baris tersebut...")
    df_clean = df_clean[df_clean['amount'] > 0]
else:
    print("Sanity check aman: Tidak ada amount bernilai negatif atau nol.")

# Mengurutkan ulang data
df_clean = df_clean.sort_values(by=['user_id', 'date']).reset_index(drop=True)

# Export ke CSV baru yang sudah bersih
cleaned_filename = 'cleaned_financial_transactions_2025.csv'
df_clean.to_csv(cleaned_filename, index=False)

print(f"\nData berhasil dibersihkan dan disimpan ke: {cleaned_filename}")
print(f"Total baris akhir: {len(df_clean):,}")

# Tampilkan hasil akhir
df_clean.head()

Sanity check aman: Tidak ada amount bernilai negatif atau nol.

Data berhasil dibersihkan dan disimpan ke: cleaned_financial_transactions_2025.csv
Total baris akhir: 34,607


,user_id,date,amount,type,category,subcategory,payment_method,description
0,USER_001,2025-01-01,10000000,income,Salary,gaji bulanan,debit,Pembayaran gaji bulanan
1,USER_001,2025-01-02,15000,expense,Food,kopi,e-wallet,Pembayaran kopi
2,USER_001,2025-01-03,35000,expense,Entertainment,bioskop,e-wallet,Pembayaran bioskop
3,USER_001,2025-01-03,15000,expense,Food,kopi,e-wallet,Pembayaran kopi
4,USER_001,2025-01-03,129500,expense,Bills,air,cash,Pembayaran air
